In [8]:
%pip install pandas openpyxl

                                              0.0/250.9 kB ? eta -:--:--
     -------                                 51.2/250.9 kB 2.7 MB/s eta 0:00:01
     ---------------                        102.4/250.9 kB 2.0 MB/s eta 0:00:01
     -------------------------------        204.8/250.9 kB 1.8 MB/s eta 0:00:01
     -------------------------------------- 250.9/250.9 kB 1.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import pandas as pd
import json

def _clean(val):
    s = str(val).strip()
    return None if s == "nan" else s

def convert_excel_to_json(excel_path, output_json_path):
    json_data = {"screening_satu_format_dan_substansi": [], "screening_dua_kategorisasi_ro": []}

    # --- 1. Sheet: Screening satu ---
    df_s1 = pd.read_excel(excel_path, sheet_name='Screening satu')
    current_section = None

    for _, row in df_s1.iterrows():
        cols = [_clean(v) for v in row.iloc[:7]]
        code, title, question, kriteria_kode, kriteria_lolos, sumber, feedback = cols

        if code in ('I', 'III', 'IV', 'V'):
            current_section = {"section_code": code, "section_title": title or kriteria_lolos, "items": []}
            json_data["screening_satu_format_dan_substansi"].append(current_section)
            continue

        if current_section and (title or question) and title != 'CHECKLIST':
            current_section["items"].append({
                "item_code": title,
                "pertanyaan": question,
                "kriteria_kode": kriteria_kode,
                "kriteria_lolos": kriteria_lolos,
                "sumber_regulasi": sumber,
                "feedback_revisi": feedback,
            })

    # --- 2. Sheet: Screening 2 ---
    df_s2 = pd.read_excel(excel_path, sheet_name='Screening 2')
    for _, row in df_s2.iterrows():
        no, question, kriteria, kategori = row.iloc[0], _clean(row.iloc[1]), _clean(row.iloc[2]), _clean(row.iloc[3])
        if pd.notna(no) and question:
            json_data["screening_dua_kategorisasi_ro"].append({
                "no": int(no),
                "pertanyaan_kriteria": question,
                "catatan_kriteria": kriteria,
                "kategori_target": kategori,
            })

    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(json_data, f, indent=2, ensure_ascii=False)

    print(f"Berhasil mengonversi Excel ke {output_json_path}!")

# Jalankan fungsi
convert_excel_to_json('Checklist Reviu Analisis (1).xlsx', 'rules_config.json')


Berhasil mengonversi Excel ke rules_config.json!
